In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import LSTM,Dense,Input
from sklearn.model_selection import train_test_split


In [3]:
with open('fra.txt','r',encoding='utf-8') as f:
    lines=f.read().split('\n')

In [4]:
lines


['Go.\tVa !\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1158250 (Wittydev)',
 'Go.\tMarche.\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8090732 (Micsmithel)',
 'Go.\tEn route !\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #8267435 (felix63)',
 'Go.\tBouge !\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #9022935 (Micsmithel)',
 'Hi.\tSalut !\tCC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #509819 (Aiji)',
 'Hi.\tSalut.\tCC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #4320462 (gillux)',
 'Run!\tCours\u202f!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #906331 (sacredceltic)',
 'Run!\tCourez\u202f!\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #906332 (sacredceltic)',
 'Run!\tPrenez vos jambes à vos cous !\tCC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #2077449 (sacredceltic)',
 'Run!\tFile !\tCC-BY 2.0 (France) Attribution: tatoeba.org #90

# prepare data

In [5]:
english_sentences=[]
french_sentences=[]

In [6]:
for line in lines[:10000]:
  parts =line.split('\t')
  if len(parts)>=2:
    eng,fra=parts[0],parts[1]
    english_sentences.append(eng)
    french_sentences.append('\t'+ fra+'\n')




In [7]:
english_sentences[0:10]

['Go.', 'Go.', 'Go.', 'Go.', 'Hi.', 'Hi.', 'Run!', 'Run!', 'Run!', 'Run!']

In [8]:
french_sentences[0:10]

['\tVa !\n',
 '\tMarche.\n',
 '\tEn route !\n',
 '\tBouge !\n',
 '\tSalut !\n',
 '\tSalut.\n',
 '\tCours\u202f!\n',
 '\tCourez\u202f!\n',
 '\tPrenez vos jambes à vos cous !\n',
 '\tFile !\n']

In [9]:
input_characters=sorted(set("".join(english_sentences)))
output_characters=sorted(set("".join(french_sentences)))

num_encoder_tokens=len(input_characters)
num_decoder_tokens=len(output_characters)

In [10]:
max_encoder_seq_length=max([len(s) for s in english_sentences])
max_decoder_seq_length=max([len(s) for s in french_sentences])

In [11]:
input_token_index=dict([(char,i) for i,char in enumerate(input_characters)])
target_token_index=dict([(char,i) for i,char in enumerate(output_characters)])
refers_target_char_index=dict((i,char) for char,i in target_token_index.items())

In [12]:
num_encoder_tokens

70

In [13]:
max_encoder_seq_length

14

In [14]:
encoder_input_data=np.zeros((len(english_sentences),max_encoder_seq_length,num_encoder_tokens),dtype='float32')

In [18]:
decoder_input_data=np.zeros((len(french_sentences),max_decoder_seq_length,num_decoder_tokens),dtype='float32')
decoder_target_data=np.zeros((len(french_sentences),max_decoder_seq_length,num_decoder_tokens),dtype='float32')

In [19]:
decoder_target_data

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.

In [21]:
for i, (input_text, target_text) in enumerate(zip(english_sentences, french_sentences)):
    for t, char in enumerate(input_text):
        encoder_input_data[i,t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t>0:
            decoder_target_data[i, t-1, target_token_index[char]] = 1.0


# Create encoder decoder model


In [26]:
# Define the encoder model
encoder_inputs = Input(shape = (None, num_encoder_tokens))
encoder_lstm = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]


In [30]:
decoder_inputs = Input(shape = (None, num_decoder_tokens))
decoder_lstm = LSTM(256, return_sequences=True, return_state= True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state = encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation= "softmax")
decoder_outputs = decoder_dense(decoder_outputs)

In [32]:
from tensorflow.keras.models import Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [33]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 70)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_4       │ (None, None, 91)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    334,848 │ input_layer[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, None,     │    356,352 │ input_layer_4[0]… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 91)  │     23,387 │ lstm_4[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 714,587 (2.73 MB)

 Trainable params: 714,587 (2.73 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.compile(
    optimizer = "rmsprop",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

In [36]:
model.fit([encoder_input_data, decoder_input_data],
decoder_target_data,
validation_split = 0.2,
epochs = 5,
batch_size = 64)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 24s 172ms/step - accuracy: 0.0346 - loss: 1.1222 - val_accuracy: 0.0502 - val_loss: 1.1072
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 22s 174ms/step - accuracy: 0.0453 - loss: 0.9797 - val_accuracy: 0.0509 - val_loss: 1.0823
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 22s 178ms/step - accuracy: 0.0461 - loss: 0.9574 - val_accuracy: 0.0544 - val_loss: 1.0569
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 23s 183ms/step - accuracy: 0.0491 - loss: 0.9573 - val_accuracy: 0.0632 - val_loss: 1.0509
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 24s 191ms/step - accuracy: 0.0527 - loss: 0.9386 - val_accuracy: 0.0560 - val_loss: 1.0486
